# Áp dụng LLM để gán lại nhãn

In [1]:
!pip install google-generativeai pandas openpyxl

In [2]:
!pip install --upgrade google-generativeai

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
Using cached google_generativeai-0.8.6-py3-none-any.whl (155 kB)
  Attempting uninstall: google-generativeai
    Found existing installation: google-generativeai 0.8.5
    Uninstalling google-generativeai-0.8.5:
      Successfully uninstalled google-generativeai-0.8.5


In [ ]:
import google.generativeai as genai

# Nhớ thay API_KEY của bạn vào
genai.configure(api_key="API_CUA_BAN")

print("Các model hỗ trợ tạo văn bản trên tài khoản của bạn:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

c:\Users\Thu Trang\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\Thu Trang\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Thu Trang\AppData\Local\Temp\ipykernel_17008\3971860397.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package 

Các model hỗ trợ tạo văn bản trên tài khoản của bạn:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-previ

In [ ]:
import pandas as pd
import google.generativeai as genai
import json
import time
import re
import math

# 1. CẤU HÌNH API
API_KEY = "API_CUA_BAN"
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('models/gemini-3.1-flash-lite')

# 2. PROMPT TEMPLATE 
SYSTEM_PROMPT = """
Bạn là chuyên gia phân tích dữ liệu phản hồi sinh viên. 
Tôi gửi một danh sách các câu phản hồi (được đánh số ID).
Hãy phân loại TỪNG CÂU vào 1 nhãn Sentiment và 1 nhãn Topic.

[ĐỊNH NGHĨA SENTIMENT]
- Positive: Khen ngợi, hài lòng, đồng tình.
- Negative: Chê bai, phàn nàn, bức xúc, góp ý sửa đổi.
- Neutral: Khách quan, bình thường, hỏi đáp.

[ĐỊNH NGHĨA TOPIC]
- Lecturer (Giảng viên): Thái độ, cách dạy, chuyên môn.
- Training (Chương trình): Nội dung môn học, slide, giáo trình, đề thi, điểm số.
- Facility (Cơ sở vật chất): Máy chiếu, wifi, phòng học.
- Others (Khác): Vấn đề khác, hoặc câu quá ngắn/vô nghĩa.

[ĐỊNH DẠNG ĐẦU RA BẮT BUỘC]
Chỉ trả về MỘT MẢNG JSON hợp lệ, KHÔNG có markdown (```json), chứa kết quả của TẤT CẢ các câu theo đúng định dạng sau:
[
    {
        "id": "id_cua_cau",
        "reasoning": "ngắn gọn 1 câu giải thích",
        "sentiment": "Positive | Negative | Neutral",
        "topic": "Lecturer | Training | Facility | Others"
    }
]
"""

def extract_json_array(text):
    text = text.strip()
    text = re.sub(r'```json\n?', '', text)
    text = re.sub(r'```\n?', '', text)
    return json.loads(text)

# 3. ĐỌC FILE VÀ CHUẨN BỊ BATCHING
input_file = r"C:\Users\Thu Trang\Downloads\Analyzing-Vietnamese-student-feedback\student-feedback-mtl\TestSet_Targeted_Relabel_Suspects.xlsx"
df = pd.read_excel(input_file)

# Test 30 câu đầu tiên xem format đã đúng chưa
df = df.head(30)

# Lọc ra những câu chưa có nhãn
unprocessed_df = df[df['LLM_Sentiment'].isna() | (df['LLM_Sentiment'].astype(str).str.strip() == '')]
print(f"Số câu cần xử lý: {len(unprocessed_df)} / Tổng số: {len(df)}")

BATCH_SIZE = 15 # Gộp 15 câu vào 1 lần gọi API
num_batches = math.ceil(len(unprocessed_df) / BATCH_SIZE)

# 4. CHẠY VÒNG LẶP THEO BATCH
for i in range(num_batches):
    start_idx = i * BATCH_SIZE
    end_idx = min((i + 1) * BATCH_SIZE, len(unprocessed_df))
    batch_df = unprocessed_df.iloc[start_idx:end_idx]
    
    print(f"\n[Lô {i+1}/{num_batches}] Đang gửi {len(batch_df)} câu cùng lúc...")
    
    # Gom 15 câu thành 1 đoạn text
    user_input = "Danh sách câu cần phân tích:\n"
    for idx, row in batch_df.iterrows():
        user_input += f"ID {idx}: {row['sentence']}\n"
        
    prompt = f"{SYSTEM_PROMPT}\n\n{user_input}"
    
    # Auto Retry cho nguyên 1 lô
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            results = extract_json_array(response.text)
            
            # Cập nhật kết quả vào DataFrame
            for res in results:
                try:
                    original_idx = int(str(res['id']).replace('ID ', '').strip())
                    df.at[original_idx, 'LLM_Reasoning'] = res.get('reasoning', '')
                    df.at[original_idx, 'LLM_Sentiment'] = res.get('sentiment', '')
                    df.at[original_idx, 'LLM_Topic']     = res.get('topic', '')
                except:
                    continue # Bỏ qua nếu LLM trả về ID sai format
                
            print(f"-> Đã xử lý thành công Lô {i+1}!")
            time.sleep(5) # Đợi 5s giữa các lô là cực kỳ an toàn
            
            # Lưu backup liên tục mỗi khi xong 1 lô
            df.to_excel(r"C:\Users\Thu Trang\Downloads\Analyzing-Vietnamese-student-feedback\student-feedback-mtl\Test_Backup.xlsx", index=False)
            break 
            
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "Quota" in error_msg:
                print(f"-> Quá tải (Thử lại {attempt+1}/{max_retries}). Đợi 30s...")
                time.sleep(30)
            else:
                print(f"-> Lỗi định dạng JSON ở Lô {i+1}, bỏ qua lô này. Chi tiết: {error_msg}")
                break

# 5. LƯU KẾT QUẢ CUỐI CÙNG
output_file = r"C:\Users\Thu Trang\Downloads\Analyzing-Vietnamese-student-feedback\student-feedback-mtl\LLM_test.xlsx"
df.to_excel(output_file, index=False)
print(f"\nHOÀN THÀNH TOÀN BỘ! File cuối cùng lưu tại: {output_file}")

Số câu cần xử lý: 30 / Tổng số: 30

[Lô 1/2] Đang gửi 15 câu cùng lúc...
-> Lỗi định dạng JSON ở Lô 1, bỏ qua lô này. Chi tiết: 403 Your project has been denied access. Please contact support.

[Lô 2/2] Đang gửi 15 câu cùng lúc...
-> Lỗi định dạng JSON ở Lô 2, bỏ qua lô này. Chi tiết: 403 Your project has been denied access. Please contact support.

HOÀN THÀNH TOÀN BỘ! File cuối cùng lưu tại: C:\Users\Thu Trang\Downloads\Analyzing-Vietnamese-student-feedback\student-feedback-mtl\LLM_test.xlsx
